<a href="https://colab.research.google.com/github/KalindiGosine/qpix-digital/blob/main/dev_journals/kgosine_larpixsim_032026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simulation Framework for Chip Networks: Update

### Introduction

This work presents an update on a simulation framework designed for networks of readout chips. The framework is intended to model an m×n network of chips together with a source chip connected to an FPGA for data collection. The goal is to provide a scalable environment in which many chips can be simulated together while preserving detailed digital behavior at the chip level.

### Program Description

The simulation program models the behavior of a two-dimensional network of readout chips, together with a source chip that interfaces with an FPGA for data collection. Each chip operates as an independent hardware simulation. Rather than treating the network as a single monolithic testbench, the framework distributes the simulation across separate processes, allowing the network to scale without linearly increasing runtime.

The framework is the same as the toy model presented on March 9th but with some expansion of sockets to accommodate 4 edge communication as well as additional state information collection.

### Socket Communication Architecture Update

The framework relies on explicit socket communication between the orchestrator and chip processes and between neighboring chips. The state information collector gathers simulation outputs for later analysis and visualization, while request/reply and push/pull socket patterns coordinate the movement of clock ticks, data requests, and state data.

The main framework update involved instantiating both input and output REQ/REP socket pairs for each of the four chip edges. Previously, chips only communicated with two chips which were explicitly defined as either upstream or downstream, meaning a chip only needed one input REQ/REP socket pair on the edge facing the upstream chip and one output REQ/REP socket pair on the edge facing the downstream chip. In this version, the edges which face upstream versus downstream chips/data paths are defined via the RTL and configuration registers within the chip, requiring each chip edge to have the ability to be both an input and output socket pair.

<img src="https://drive.google.com/uc?export=view&id=1MJhQyi86o9oOZj4NoatvKOkI8uvTObW3" width="500">


<img src="https://drive.google.com/uc?export=view&id=1BV3QBg7p9NwiplVXOnvaZ0R4dPJ97H5m" width="500">




### LArPix v3b Digital Core as a Test Case

As the first non-trivial test of this framework, the LArPix v3b digital core was used as the input RTL to the simulator. Only minimal changes were required to the overall framework. The primary modifications were:

1.   additional mechanisms for collecting chip state information
2.   additional nng sockets to enable communication between chip processes in all four directions

Note that the framework is designed with the idea that any generic RTL could be simulated so as to be a useful intermediate environment between high-level routing studies and true hardware deployment.


### 3×5 LArPix Network Case: Chip ID Assignment

A first network-level study was performed on a 3×5 LArPix array. In the startup configuration (defined in the RTL), all chips begin with:


*   chip_id = 1
*   all four RX lanes enabled
*   no TX lanes enabled


<img src="https://drive.google.com/uc?export=view&id=12d2AZV5r25G4W3pHsmCn2EjWsDgkqLnW" width="500">

From this initial state, a schedule of configuration write packets is generated and sent by the FPGA into the source chip. The configuration proceeds by cycling through a repeating pattern:

*  enable the downstream TX lane
*  assign a chip ID
*  enable the upstream TX lane.

While this step of assigning chip ids is necessary for all future tests on a chip network, it also serves as a first test of sending configuration write and read packets through the network and seeing the state of chips update.


<img src="https://drive.google.com/uc?export=view&id=12Mdg-K1WISrg8dLq-P6B4lVLSghYY2sW" width="500">


### 3×5 LArPix Network Case: Charge Injection

A second study on the same 3×5 array examines data generation from charge injection. In this case, the disable masks are removed from all channels in Chip 14, and natural triggering is enabled. A charge of −5×e−15 is then injected into each of the 64 channels of that chip. The simulation then follows the resulting data packets as they are generated and begin to leave Chip 14. FIFO occupancy is monitored as part of the analysis.

<img src="https://drive.google.com/uc?export=view&id=1xli92B8SSjDi05yNglcMArZWd5dp-NkW" width="500">

This case shows how the framework can connect physical stimulus, channel-level activity, and network-level communication in one simulation environment. Rather than merely reporting that packets were produced, the simulation resolves the timing and buffering behavior through the actual digital architecture and holds every bit of the generated data packet travelling through the network.

The charge injection into the channels occurs at simulation tick 9800. Within about eight ticks, all 64 channels have generated a data packet. Each of these packets first enters a local channel-level FIFO, causing the occupancy of each local FIFO to rise to one.

<img src="https://drive.google.com/uc?export=view&id=1n0hP4081woiD2UJ7liixjUnIXNmCdoU-" width="500">

The RTL defines a round-robin arbiter that pulls data packets from the local FIFOs into the chip FIFO. The transfer begins from Channel 0. As packets are removed from their local FIFOs, local occupancy returns to zero while chip FIFO occupancy increases. This process repeats such that, on each clock tick, one packet is removed from a local FIFO and placed into the chip FIFO, and the chip FIFO occupancy rises by one on each clock tick.

This is a useful example of the kind of information the framework can reveal. The simulation does not only show that data is present; it reveals how arbitration and queueing operate cycle by cycle, making it possible to analyze the detailed timing of packet movement through the digital design.

#### FIFO Dequeuing into the Hydra TX Path

Once packets have entered the chip FIFO, they do not leave continuously every clock tick. Instead, the simulation shows that one packet exits the chip FIFO every 69 ticks and enters the Hydra TX data path and then the selected UART transmitter. The plotted quantity is the tick on which the packet exits the FIFO, not the tick on which it leaves the chip entirely.

<img src="https://drive.google.com/uc?export=view&id=1TvQpFWWwI2HHa7xX8G0LaBWHX3mfleZk" width="500">

<img src="https://drive.google.com/uc?export=view&id=1KvwCjH_I5BMvTx1td4MphvCmEHlFt7yE" width="500">


This interval is rooted in the structure of the transmitted packet and the transmit path timing. Each data packet is 64 bits, and it is sent serially with the addition of one start bit and one stop bit, and roughly three additional ticks for the Hydra TX logic to:

*  change state from IDLE to TX_GET_FIFO,
*  perform the FIFO read
*  load the packet into the UART TX block.

Together, these account for the 69-tick interval between successive dequeues from the chip FIFO. This illustrates how the framework can connect FIFO behavior, control-state transitions, and serialized transmission timing into a single analysis.

### 15×15 LArPix Network Case

This larger case supports one of the main claims of the framework: it can scale to arrays far larger than those that are convenient to simulate in a traditional RTL testbench while still preserving bit- and clock-level digital behavior. These networks also highlight opportunities for improvement on routing configurations as we can track data packet loss from charge events as packets move towards the source chip. These large network arrays with charge injection are the prime target for connecting the input of these simulations to the expected charge deposition as a function of position on a plane from a physics simulator.

<img src="https://drive.google.com/uc?export=view&id=1nYLzqSda5ARVyMxJu9luv8Vdm70_OjOF" width="500">




### Comparison to Existing Network Simulators

The framework was compared to two existing simulation tools: BookSim and FireSim. The purpose of the comparison is to clarify what design space this work addresses.

<img src="https://drive.google.com/uc?export=view&id=1xpgcLXEZ8_Oo0bR3BaJoO0zf0qpjG0QB" width="300">

#### BookSim

BookSim is a cycle-accurate network simulator dedicated to studying network performance. It focuses on the movement of flow-control units (flits) rather than bits, and it is primarily concerned with information such as queueing, arbitration, bandwidth, latency, and congestion. In BookSim, the node is represented as an abstraction built from routers, buffers, and channels. It is therefore well suited to large-scale network studies with abstract node models.

#### FireSim

FireSim is a cycle-accurate FPGA-accelerated simulator for studying RTL-defined systems at scale. It focuses on system-level behavior, timing, and communication between simulated nodes. In this view, the link between nodes is modeled so that communication behavior is captured without bit-level information. FireSim is intended for cases where the simulated nodes are large RTL-defined computing blocks such as SoCs.

#### Position of the Present Framework

The presented framework is designed for a different regime. The chip designs of interest are too large to simulate as a large network within a conventional SystemVerilog testbench. At the same time, the individual chips are small enough that the simulation does not need to give up bit-level or clock-edge accuracy in order to study the network. The framework therefore fills a gap between abstract network simulators and large-node system simulators. It also opens the possibility of mixed-signal accuracy through Verilator-based co-simulation with charge input.

### Project Integration and Role in Design Studies

The framework is intended to support ongoing work on the optimization of topology and routing in chip networks. In this role, it acts as a bridge between high-level routing design studies and bit-level hardware behavior. In a LArPix-like system, routing or configuration designs may originate from broader network studies, but they must eventually be implemented through real packet transmissions and chip-level configuration behavior. This framework provides a way to validate those ideas closer to the hardware implementation stage and therefore move toward pre-fabrication verification.

### Next Steps

The framework already supports:

* simulation network scaling
* RTL-accurate behavior
* simple design parameter studies
* easy drop in of non-trivial RTL-defined chip designs

The next major directions are:

* replacing current analog front-end approximations defined in software with transistor-level SPICE simulation of analog components
* using detector simulation charge data as input to test network response to real charge track events


